> ### **Salesperson Clean**

In [0]:
bronze_df = spark.table("accenture_final_project.bronze_layer.salesperson")
display(bronze_df)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, to_date, initcap, trim
from pyspark.sql.types import IntegerType, StringType

In [0]:
import pyspark.sql.functions as F
# 1. Read from Bronze layer
bronze_df_salesperson = spark.table("accenture_final_project.bronze_layer.salesperson")

# 2. Schema Enforcement / Transformation
df_salesperson_raw = (bronze_df_salesperson
    .select(
        col("EmployeeKey").cast(IntegerType()),
        col("EmployeeID").cast(IntegerType()),
        col("Salesperson").cast(StringType()),
        col("Title").cast(StringType()),
        col("UPN").cast(StringType())
    )
)

In [0]:
# Silver Layer Transformations
df_salesperson_silver = (df_salesperson_raw
    # Basic standardization of text fields: Trim whitespaces
    .withColumn("Salesperson", F.trim(F.col("Salesperson")))
    .withColumn("Title", F.trim(F.col("Title")))
    .withColumn("UPN", F.lower(F.trim(F.col("UPN"))))
    .dropDuplicates()
    .drop("_rescued_data")
)

In [0]:
display(df_salesperson_silver)

In [0]:
# Check the data types of all columns in the DataFrame
df_salesperson_silver.printSchema()

# Display the first 10 rows to visually inspect the values (e.g., date formats, currency symbols)
df_salesperson_silver.show(10, truncate=False)

In [0]:
# --- OUTLIER & ANOMALY HANDLING ---
# Keep only valid records: salesperson must be positive (no negative salesperson) and not null
df_salesperson_clean = df_salesperson_silver.filter(
    (F.col("Target").isNotNull()) & 
    (F.col("Target") > 0) &
    (F.col("TargetMonth").isNotNull())
)

## **Transform to DELTA**

In [0]:
# Targets DELTA
df_salesperson_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("accenture_final_project.silver_layer.salesperson")